## Setup

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from src.py_src import util
from tqdm import tqdm
from src.py_src.models import SolarFlarePredictionModel
import numpy as np

pd.set_option('mode.copy_on_write', True)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
load_dotenv()

EVENTS_TREATED_GLOBAL_PATH = os.getenv("EVENTS_TREATED_GLOBAL_PATH")
XRAY_TREATED_GLOBAL_PATH = os.getenv("XRAY_TREATED_GLOBAL_PATH")

data_begin_year = 2010
data_end_year = 2024
data_year_range = range(data_begin_year, data_end_year + 1)

cols_names = ['xl']

## Reading CSVs

In [ ]:
data = {
    'events': {},
    'xrays': {}
}

In [ ]:
missing_files = []
for y in range(1983, 1995+1):
    f = missing_files.append(os.path.join(EVENTS_TREATED_GLOBAL_PATH, os.path.join(str(y), f"{y}_events.csv")))
    missing_files.append(os.path.join(EVENTS_TREATED_GLOBAL_PATH, os.path.join(str(y), f"{y}_DSD.csv")))

In [ ]:
for y in tqdm(data_year_range):
    events_year_dir = os.path.join(EVENTS_TREATED_GLOBAL_PATH, str(y))
    xrays_year_dir = os.path.join(XRAY_TREATED_GLOBAL_PATH, str(y))

    try:
        data['events'][y] = pd.read_parquet(os.path.join(events_year_dir, f"{y}_xra_events_global.parquet")
                                               ).rename(columns={'date':'ds'})
    except FileNotFoundError as e:
        data['events'][y] = pd.DataFrame()
        if e.filename not in missing_files: print(e)

    try:
        data['xrays'][y] = pd.read_csv(os.path.join(xrays_year_dir, f"{y}_xrays.csv"),
                                       parse_dates=['ds'],
                                       index_col='ds')
        data['xrays'][y] = data['xrays'][y].tz_localize('UTC')
        data['xrays'][y] = data['xrays'][y].asfreq('1min')

    except FileNotFoundError as e:
        data['xrays'][y] = pd.DataFrame()
        if e.filename not in missing_files: print(e)

### DFs To Slide

In [ ]:
def get_cols(year: int) -> list[str]:
    # return ['xs','xl'] if year < 2020 else ['xrsa_flux','xrsb_flux']
    return ['xl'] if year < 2020 else ['xrsb_flux']

In [ ]:
xrays_to_slide_list = []
events_to_slide_list = []

for y in tqdm(data_year_range):
    cols = get_cols(y)
    df_xrays = data['xrays'][y][[c for c in cols]]
    if cols != cols_names:
        df_xrays = df_xrays.rename(columns=dict(zip(cols, cols_names)))

    xrays_to_slide_list.append(df_xrays)

    df_events = data['events'][y]
    events_to_slide_list.append(df_events)

xrays_to_slide = pd.concat(xrays_to_slide_list).sort_index()
events_to_slide = pd.concat(events_to_slide_list).sort_values('begin')

In [ ]:
events_to_slide

### Ground Truth

In [ ]:
def create_ground_truth(events: pd.DataFrame, indexes: pd.DatetimeIndex) -> pd.DataFrame:
    begin_arr = events['begin'].to_numpy()
    end_arr = events['end'].to_numpy()
    class_arr = events['class_numeric'].to_numpy()

    is_nat = pd.isna(events['end']).to_numpy()
    valid_mask = ~is_nat

    durations_ns = (end_arr[valid_mask] - begin_arr[valid_mask]).astype('timedelta64[ns]')
    durations_min = durations_ns.astype('timedelta64[m]').astype(float)

    unique_classes = np.unique(class_arr)
    mean_durations = {}

    for c in unique_classes:
        c_mask = (class_arr[valid_mask] == c)
        if np.any(c_mask):
            mean_durations[c] = np.mean(durations_min[c_mask])
        else:
            mean_durations[c] = 0.0

    end_imputed = end_arr.copy()
    for c in unique_classes:
        impute_mask = is_nat & (class_arr == c)
        if np.any(impute_mask):
            offset = np.timedelta64(int(mean_durations[c]), 'm')
            end_imputed[impute_mask] = begin_arr[impute_mask] + offset

    sort_idx = np.argsort(class_arr)
    begin_sorted = begin_arr[sort_idx]
    end_imputed_sorted = end_imputed[sort_idx]
    class_sorted = class_arr[sort_idx]

    gt_classes = np.zeros(len(indexes), dtype=int)
    idx_values = indexes.to_numpy()

    for b, _e, c in zip(begin_sorted, end_imputed_sorted, class_sorted):
        start_pos = np.searchsorted(idx_values, b, side='left')
        end_pos = np.searchsorted(idx_values, _e, side='right')

        gt_classes[start_pos:end_pos] = c

    return pd.DataFrame(gt_classes, index=indexes, columns=['current_class'])

In [ ]:
events_to_slide['begin'] = pd.to_datetime(events_to_slide['begin'], utc=True)
events_to_slide['max'] = pd.to_datetime(events_to_slide['max'], utc=True)
events_to_slide['end'] = pd.to_datetime(events_to_slide['end'], utc=True)

df_ground_truth = create_ground_truth(events_to_slide, xrays_to_slide.index)

percentages = df_ground_truth['current_class'].value_counts(normalize=True)
for value_label, proportion in percentages.items():
    print(f"{value_label} -> {proportion * 100:.2f}%")

## Showing DFs

In [ ]:
xrays_to_slide

In [ ]:
events_to_slide

In [ ]:
percentages = events_to_slide['class'].value_counts(normalize=True)
for value_label, proportion in percentages.items():
    print(f"{value_label} -> {proportion * 100:.2f}%")
print(len(events_to_slide))

## Slided DataFrames

In [ ]:
df_features = SolarFlarePredictionModel.generate_xray_features(xrays_to_slide, events_to_slide)
df_target = SolarFlarePredictionModel.generate_target(xrays_to_slide, events_to_slide)

df_slided = pd.concat([df_features, df_target], axis=1, copy=True)

df_slided = df_slided.dropna()

In [ ]:
max_window_duration = pd.to_timedelta('72h')
cutoff_time = df_slided.index[-1] - max_window_duration
df_slided = df_slided[df_slided.index <= cutoff_time]

In [ ]:
df_slided

In [ ]:
df_slided.shape

## Exporting

In [ ]:
slided_dfs_path = os.getenv("XRAY_SLIDED_PATH")

df_slided.to_parquet(os.path.join(slided_dfs_path, "xray_slided.parquet"))